# Fraud Detection Model — End-to-End Analysis

**Author:** Fraud Model Analytics  
**Last Updated:** 2024-Q4  
**Model Version:** v2.3  

---

## Overview

This notebook demonstrates the full fraud detection model lifecycle, from exploratory data analysis through model deployment readiness. The workflow mirrors production processes used in banking fraud operations, including:

- **Data extraction & quality checks** — validating transaction data integrity
- **Exploratory Data Analysis (EDA)** — understanding fraud patterns and distributions
- **Feature engineering** — behavioral biometrics, device profiling, velocity signals
- **Model development** — baseline (Logistic Regression) and primary (XGBoost) models
- **Model evaluation & KPI reporting** — detection rate, FPR, fraud loss prevented
- **SHAP explainability** — communicating model decisions to fraud operations and audit
- **Threshold optimization** — balancing detection coverage vs. false positive burden

### Data Sources
| Source | Description |
|--------|-------------|
| Core banking transaction warehouse | Transaction amounts, channels, merchant categories |
| Behavioral biometrics platform (BioCatch) | Session duration, typing speed, mouse patterns |
| Device intelligence (LexisNexis ThreatMetrix) | Device fingerprint, IP risk score, VPN detection |
| LexisNexis Risk Solutions API | Composite identity and network risk scores |
| Fraud case management system | Confirmed fraud labels (chargeback resolution) |

> **Note:** All data in this notebook is **synthetic** and generated to reflect realistic fraud patterns. No real customer data is used.

---
## 0. Setup & Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report
)
from xgboost import XGBClassifier
import shap

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded ✓')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')

In [ ]:
# Generate synthetic dataset (run data/generate_synthetic_data.py first, or run inline)
import sys
sys.path.insert(0, '../data')
sys.path.insert(0, '../src')

from generate_synthetic_data import generate_account_pool, simulate_transactions

accounts = generate_account_pool(n_accounts=8000)
df_raw   = simulate_transactions(accounts)

print(f'Dataset shape  : {df_raw.shape}')
print(f'Fraud rate     : {df_raw["is_fraud"].mean()*100:.2f}%')
print(f'Date range     : {df_raw["transaction_date"].min()} — {df_raw["transaction_date"].max()}')
df_raw.head(3)

---
## 1. Data Quality & Integrity Checks

Before any modeling, we validate data completeness, type consistency, and flag any anomalies that could bias model training.

In [ ]:
# Missing value analysis
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
dq_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
dq_report = dq_report[dq_report['missing_count'] > 0]

if dq_report.empty:
    print('✓ No missing values detected')
else:
    print('Missing values found:')
    display(dq_report)

# Duplicate transaction IDs
dups = df_raw['transaction_id'].duplicated().sum()
print(f'\nDuplicate transaction IDs : {dups}')

# Amount sanity check
neg_amounts = (df_raw['amount'] <= 0).sum()
print(f'Non-positive amounts       : {neg_amounts}')

# Class distribution
print(f'\nLabel distribution:')
print(df_raw['is_fraud'].value_counts().rename({0: 'Legitimate', 1: 'Fraud'}).to_string())

---
## 2. Exploratory Data Analysis

### 2.1 Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution by fraud label
for label, color, name in [(0, '#457B9D', 'Legitimate'), (1, '#E63946', 'Fraud')]:
    subset = df_raw[df_raw['is_fraud'] == label]['amount'].clip(upper=5000)
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=f'{name} (n={len(subset):,})', density=True)

axes[0].set_xlabel('Transaction Amount ($)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Transaction Amount Distribution by Label', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Amount statistics table
amount_stats = df_raw.groupby('is_fraud')['amount'].describe().round(2)
amount_stats.index = ['Legitimate', 'Fraud']
print('Amount statistics by label:')
display(amount_stats)

# Box plot
df_plot = df_raw.copy()
df_plot['label'] = df_plot['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})
df_plot['amount_clipped'] = df_plot['amount'].clip(upper=3000)
df_plot.boxplot(column='amount_clipped', by='label', ax=axes[1],
                boxprops=dict(color='#333333'),
                medianprops=dict(color='#E63946', linewidth=2))
axes[1].set_title('Amount Distribution (clipped at $3,000)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Amount ($)', fontsize=11)
plt.suptitle('')
plt.tight_layout()
plt.show()

### 2.2 Fraud Rate by Time of Day and Channel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by hour
hourly = df_raw.groupby('transaction_hour').agg(
    fraud_rate=('is_fraud', 'mean'),
    txn_count=('is_fraud', 'count')
).reset_index()

ax1 = axes[0]
bars = ax1.bar(hourly['transaction_hour'], hourly['fraud_rate'] * 100,
               color=['#E63946' if r > hourly['fraud_rate'].mean() else '#457B9D'
                      for r in hourly['fraud_rate']],
               alpha=0.85, edgecolor='white')
ax1.axhline(hourly['fraud_rate'].mean() * 100, color='black',
            linestyle='--', lw=1.5, label='Overall avg')
ax1.set_xlabel('Hour of Day (24h)', fontsize=11)
ax1.set_ylabel('Fraud Rate (%)', fontsize=11)
ax1.set_title('Fraud Rate by Hour of Day', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_xticks(range(0, 24, 2))

# Fraud rate by channel
channel_stats = df_raw.groupby('channel').agg(
    fraud_rate=('is_fraud', 'mean'),
    volume=('is_fraud', 'count'),
    fraud_count=('is_fraud', 'sum')
).reset_index().sort_values('fraud_rate', ascending=True)

axes[1].barh(channel_stats['channel'], channel_stats['fraud_rate'] * 100,
             color='#E63946', alpha=0.8)
for i, (_, row) in enumerate(channel_stats.iterrows()):
    axes[1].text(row['fraud_rate'] * 100 + 0.05, i,
                 f"{row['fraud_rate']*100:.2f}%  (n={row['fraud_count']:,})",
                 va='center', fontsize=9)
axes[1].set_xlabel('Fraud Rate (%)', fontsize=11)
axes[1].set_title('Fraud Rate by Transaction Channel', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, channel_stats['fraud_rate'].max() * 100 * 1.4)

plt.tight_layout()
plt.show()

print('\nChannel summary:')
display(channel_stats.sort_values('fraud_rate', ascending=False).reset_index(drop=True))

### 2.3 Behavioral Biometrics: Fraud vs. Legitimate Signal Separation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Behavioral Biometrics — Fraud vs. Legitimate Signal Separation',
             fontsize=13, fontweight='bold', y=1.01)

biometric_features = [
    ('session_duration_sec', 'Session Duration (seconds)', True, 300),
    ('typing_speed_wpm',     'Typing Speed (WPM)',          False, 120),
    ('mouse_linearity_score','Mouse Linearity Score (0–1)', False, None),
    ('login_attempt_count',  'Login Attempt Count',         False, None),
]

for ax, (col, label, log_scale, clip_val) in zip(axes.flatten(), biometric_features):
    for fraud_val, color, name in [(0, '#457B9D', 'Legitimate'), (1, '#E63946', 'Fraud')]:
        data = df_raw[df_raw['is_fraud'] == fraud_val][col]
        if clip_val:
            data = data.clip(upper=clip_val)
        ax.hist(data, bins=50, alpha=0.6, color=color,
                label=f'{name}', density=True)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    if log_scale:
        ax.set_yscale('log')

plt.tight_layout()
plt.show()

# Statistical summary
print('Biometric means by fraud label:')
biometric_cols = ['session_duration_sec', 'typing_speed_wpm',
                  'mouse_linearity_score', 'copy_paste_detected', 'login_attempt_count']
summary = df_raw.groupby('is_fraud')[biometric_cols].mean().round(3)
summary.index = ['Legitimate', 'Fraud']
display(summary)

### 2.4 Device Profiling: IP Risk and Device Age

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# IP Risk Score distribution
for label, color, name in [(0, '#457B9D', 'Legitimate'), (1, '#E63946', 'Fraud')]:
    axes[0].hist(df_raw[df_raw['is_fraud']==label]['ip_risk_score'],
                 bins=40, alpha=0.65, color=color, label=name, density=True)
axes[0].set_xlabel('IP Risk Score (0–100)', fontsize=10)
axes[0].set_ylabel('Density', fontsize=10)
axes[0].set_title('LexisNexis IP Risk Score', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

# Device age distribution
device_age_clip = df_raw['device_age_days'].clip(upper=365)
for label, color, name in [(0, '#457B9D', 'Legitimate'), (1, '#E63946', 'Fraud')]:
    axes[1].hist(device_age_clip[df_raw['is_fraud']==label],
                 bins=40, alpha=0.65, color=color, label=name, density=True)
axes[1].set_xlabel('Device Age (days, clipped at 365)', fontsize=10)
axes[1].set_ylabel('Density', fontsize=10)
axes[1].set_title('Device Age at Transaction', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)

# VPN / Proxy flags
vpn_by_fraud = df_raw.groupby('is_fraud')['vpn_proxy_detected'].mean() * 100
vpn_by_fraud.index = ['Legitimate', 'Fraud']
vpn_by_fraud.plot(kind='bar', ax=axes[2], color=['#457B9D', '#E63946'],
                  alpha=0.85, edgecolor='white')
for i, v in enumerate(vpn_by_fraud):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[2].set_title('VPN/Proxy Detected Rate', fontsize=11, fontweight='bold')
axes[2].set_ylabel('% Transactions with VPN/Proxy', fontsize=10)
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 2.5 Feature Correlation with Fraud Label

In [ ]:
numeric_cols = [
    'amount', 'amount_to_limit_ratio', 'ip_risk_score', 'lexisnexis_risk_score',
    'session_duration_sec', 'typing_speed_wpm', 'mouse_linearity_score',
    'copy_paste_detected', 'login_attempt_count', 'device_age_days',
    'vpn_proxy_detected', 'txn_count_24h', 'distinct_merchants_24h',
    'geo_mismatch', 'is_card_not_present', 'account_age_days',
    'prior_disputes', 'is_weekend'
]

correlations = df_raw[numeric_cols + ['is_fraud']].corr()['is_fraud'].drop('is_fraud')
correlations = correlations.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E63946' if v > 0 else '#457B9D' for v in correlations]
correlations.plot(kind='barh', ax=ax, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson Correlation with Fraud Label', fontsize=11)
ax.set_title('Feature Correlation with Fraud (sorted by |correlation|)',
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Top positive fraud correlates:')
print(correlations[correlations > 0].head(8).to_string())
print('\nTop negative fraud correlates (protective factors):')
print(correlations[correlations < 0].tail(5).to_string())

---
## 3. Feature Engineering

We engineer composite signals that capture fraud patterns more precisely than raw features alone.

In [ ]:
from feature_engineering import build_features, FEATURE_COLUMNS

# Build full feature matrix
df_features, encoder = build_features(df_raw, fit_encoder=True)
y = df_raw['is_fraud'].astype(int)

print(f'Feature matrix shape : {df_features.shape}')
print(f'Features used        : {len(FEATURE_COLUMNS)}')
print(f'\nFeature groups:')
print('  Core transaction   : amount, ratios, channel, account_age')
print('  Time               : hour_sin/cos, is_night_txn, is_weekend')
print('  Amount deviation   : amount_zscore, outlier_flag')
print('  Velocity           : txn_count_1h/24h, amount_sum, merchant_diversity')
print('  Behavioral biom.   : session_dur, typing_speed, mouse_linearity, bot_signal')
print('  Device profiling   : device_age, ip_risk_norm, vpn, new_device_high_risk')
print('  LexisNexis         : ln_risk_score_norm, geo_mismatch')

df_features.describe().round(3)

In [ ]:
# Validate key engineered features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

new_features = ['bot_signal', 'new_device_high_ip_risk', 'cnp_with_anonymizer']
labels = ['Bot Signal\n(short session + copy-paste)',
          'New Device + High IP Risk',
          'CNP + VPN/Proxy']

for ax, feat, label in zip(axes, new_features, labels):
    fraud_rate_feat = df_raw.groupby(df_features[feat])['is_fraud'].mean() * 100
    bars = ax.bar(['Not Triggered', 'Triggered'],
                  [fraud_rate_feat.get(0, 0), fraud_rate_feat.get(1, 0)],
                  color=['#457B9D', '#E63946'], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, f'{h:.1f}%',
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('Fraud Rate (%)', fontsize=10)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(fraud_rate_feat.values) * 1.3)

plt.suptitle('Engineered Feature Fraud Rate Lift', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Model Development

### 4.1 Train-Test Split (Time-Aware)

We use a **time-based split** rather than random split to prevent data leakage and simulate real deployment conditions where the model scores future transactions.

In [ ]:
# Sort by time and split
df_sorted   = df_raw.sort_values('transaction_timestamp').reset_index(drop=True)
y_sorted    = y.loc[df_sorted.index].reset_index(drop=True)
feat_sorted = df_features.loc[df_sorted.index].reset_index(drop=True)

cutoff      = int(len(df_sorted) * 0.80)
X_train     = feat_sorted.iloc[:cutoff]
X_test      = feat_sorted.iloc[cutoff:]
y_train     = y_sorted.iloc[:cutoff]
y_test      = y_sorted.iloc[cutoff:]
amounts_test = df_sorted['amount'].iloc[cutoff:].reset_index(drop=True)

print(f'Training set : {len(X_train):,} transactions ({y_train.sum():,} fraud)')
print(f'Test set     : {len(X_test):,} transactions ({y_test.sum():,} fraud)')
print(f'Train fraud rate : {y_train.mean()*100:.2f}%')
print(f'Test fraud rate  : {y_test.mean()*100:.2f}%')

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'\nClass imbalance ratio (scale_pos_weight) : {scale_pos_weight:.1f}x')

### 4.2 Baseline: Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    C=0.1,
    class_weight='balanced',
    solver='lbfgs',
    max_iter=1000,
    random_state=RANDOM_STATE
)

lr_model.fit(X_train, y_train)
lr_probs = lr_model.predict_proba(X_test)[:, 1]

lr_roc  = roc_auc_score(y_test, lr_probs)
lr_prauc = average_precision_score(y_test, lr_probs)

print(f'Logistic Regression:')
print(f'  ROC-AUC  : {lr_roc:.4f}')
print(f'  PR-AUC   : {lr_prauc:.4f}')

# Top features by coefficient magnitude
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#E63946' if c > 0 else '#457B9D' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Logistic Regression — Top Coefficients (fraud risk direction)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Coefficient Value', fontsize=10)
plt.tight_layout()
plt.show()

### 4.3 Primary Model: XGBoost

XGBoost is the production model. It handles non-linear interactions, missing values, and class imbalance (via `scale_pos_weight`) better than logistic regression.

In [ ]:
# Production-tuned hyperparameters (from Optuna optimization — see model_training.py)
xgb_params = {
    'n_estimators':     450,
    'max_depth':        5,
    'learning_rate':    0.05,
    'subsample':        0.80,
    'colsample_bytree': 0.75,
    'min_child_weight': 8,
    'reg_alpha':        0.5,
    'reg_lambda':       2.0,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric':      'aucpr',
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
    'verbosity':        0,
}

xgb_model = XGBClassifier(**xgb_params)

# 5-fold cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(xgb_model, X_train, y_train,
                             cv=cv, scoring='average_precision', n_jobs=-1)

print(f'5-Fold CV PR-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {[round(s, 4) for s in cv_scores]}')

# Train final model
xgb_model.fit(X_train, y_train)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

xgb_roc   = roc_auc_score(y_test, xgb_probs)
xgb_prauc = average_precision_score(y_test, xgb_probs)

print(f'\nXGBoost (test set):')
print(f'  ROC-AUC  : {xgb_roc:.4f}')
print(f'  PR-AUC   : {xgb_prauc:.4f}')

---
## 5. Model Evaluation & KPI Reporting

### 5.1 ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison: ROC & Precision-Recall Curves',
             fontsize=13, fontweight='bold', y=1.01)

models_eval = {
    'Logistic Regression': lr_probs,
    'XGBoost': xgb_probs
}
colors_eval = ['#457B9D', '#E63946']

for (name, probs), color in zip(models_eval.items(), colors_eval):
    fpr_arr, tpr_arr, _ = roc_curve(y_test, probs)
    roc_auc = roc_auc_score(y_test, probs)
    axes[0].plot(fpr_arr, tpr_arr, lw=2.5, color=color,
                 label=f'{name} (AUC={roc_auc:.3f})')

    prec_arr, rec_arr, _ = precision_recall_curve(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)
    axes[1].plot(rec_arr, prec_arr, lw=2.5, color=color,
                 label=f'{name} (PR-AUC={pr_auc:.3f})')

axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Detection Rate)', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1,
                label=f'No-skill ({y_test.mean()*100:.1f}%)')
axes[1].set_xlabel('Recall (Detection Rate)', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Precision-Recall Curve', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Fraud KPI Dashboard

In [ ]:
THRESHOLD = 0.50

def fraud_kpi_report(y_true, y_prob, amounts, model_name, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(((y_pred==1) & (y_true==1)).sum())
    fp = int(((y_pred==1) & (y_true==0)).sum())
    tn = int(((y_pred==0) & (y_true==0)).sum())
    fn = int(((y_pred==0) & (y_true==1)).sum())

    detection_rate = tp / max(tp+fn, 1)
    fpr            = fp / max(fp+tn, 1)
    precision      = tp / max(tp+fp, 1)

    fraud_exposure    = amounts[y_true==1].sum()
    fraud_caught_usd  = amounts[(y_pred==1) & (y_true==1)].sum()
    fraud_escaped_usd = amounts[(y_pred==0) & (y_true==1)].sum()
    # Assume 85% recovery rate on caught fraud
    fraud_loss_prevented = fraud_caught_usd * 0.85

    kpis = {
        'Model':                model_name,
        'Threshold':            threshold,
        'ROC-AUC':              round(roc_auc_score(y_true, y_prob), 4),
        'PR-AUC':               round(average_precision_score(y_true, y_prob), 4),
        'Detection Rate':       f'{detection_rate*100:.2f}%',
        'Precision':            f'{precision*100:.2f}%',
        'False Positive Rate':  f'{fpr*100:.4f}%',
        'FP per TP':            round(fp/max(tp,1), 1),
        'TP':                   tp,
        'FP':                   fp,
        'TN':                   tn,
        'FN':                   fn,
        'Alerts Generated':     tp+fp,
        'Fraud Exposure ($)':   f'${fraud_exposure:,.2f}',
        'Fraud Caught ($)':     f'${fraud_caught_usd:,.2f}',
        'Fraud Escaped ($)':    f'${fraud_escaped_usd:,.2f}',
        'Loss Prevented ($)':   f'${fraud_loss_prevented:,.2f}',
        'Dollar Detection':     f'{fraud_caught_usd/max(fraud_exposure,1)*100:.2f}%',
    }
    return kpis

lr_kpis  = fraud_kpi_report(y_test, lr_probs,  amounts_test, 'Logistic Regression', THRESHOLD)
xgb_kpis = fraud_kpi_report(y_test, xgb_probs, amounts_test, 'XGBoost',             THRESHOLD)

kpi_df = pd.DataFrame([lr_kpis, xgb_kpis]).set_index('Model').T
print('=== FRAUD MODEL KPI REPORT ===')
display(kpi_df)

### 5.3 Threshold Sensitivity Analysis

Model owners and fraud operations use this to select the operating threshold that best balances detection coverage against false positive review burden.

In [ ]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
rows = []

for t in thresholds:
    y_pred = (xgb_probs >= t).astype(int)
    tp = int(((y_pred==1)&(y_test==1)).sum())
    fp = int(((y_pred==1)&(y_test==0)).sum())
    tn = int(((y_pred==0)&(y_test==0)).sum())
    fn = int(((y_pred==0)&(y_test==1)).sum())

    fraud_exposure    = amounts_test[y_test==1].sum()
    fraud_caught_usd  = amounts_test[(y_pred==1) & (y_test.values==1)].sum()

    rows.append({
        'Threshold':         t,
        'Detection Rate %':  round(tp/max(tp+fn,1)*100, 2),
        'Precision %':       round(tp/max(tp+fp,1)*100, 2),
        'FPR %':             round(fp/max(fp+tn,1)*100, 4),
        'FP per TP':         round(fp/max(tp,1), 1),
        'Alerts':            tp+fp,
        'TP':                tp,
        'FP':                fp,
        'FN':                fn,
        'Dollar Detection %': round(fraud_caught_usd/max(fraud_exposure,1)*100, 2),
    })

sens_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('XGBoost — Threshold Sensitivity Analysis', fontsize=13, fontweight='bold')

axes[0].plot(sens_df['Threshold'], sens_df['Detection Rate %'], 'o-',
             color='#E63946', lw=2, label='Detection Rate (%)')
ax2 = axes[0].twinx()
ax2.plot(sens_df['Threshold'], sens_df['FPR %'], 's--',
         color='#457B9D', lw=2, label='False Positive Rate (%)')
axes[0].set_xlabel('Threshold', fontsize=11)
axes[0].set_ylabel('Detection Rate (%)', color='#E63946', fontsize=10)
ax2.set_ylabel('False Positive Rate (%)', color='#457B9D', fontsize=10)
axes[0].set_title('Detection Rate vs. FPR Trade-off', fontsize=11, fontweight='bold')
axes[0].axvline(THRESHOLD, color='black', ls=':', lw=1.5, label=f'Current threshold={THRESHOLD}')
lines, labels = axes[0].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[0].legend(lines+lines2, labels+labels2, fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].bar([str(t) for t in sens_df['Threshold']], sens_df['TP'],
            color='#E63946', alpha=0.85, label='True Positives')
axes[1].bar([str(t) for t in sens_df['Threshold']], sens_df['FP'],
            bottom=sens_df['TP'], color='#457B9D', alpha=0.7, label='False Positives')
axes[1].set_xlabel('Threshold', fontsize=11)
axes[1].set_ylabel('Alert Count', fontsize=11)
axes[1].set_title('Alert Composition by Threshold', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nThreshold sensitivity table:')
display(sens_df)

---
## 6. SHAP Explainability

SHAP (SHapley Additive exPlanations) values explain the contribution of each feature to individual predictions. This is critical for:
- **Fraud operations**: understanding why a transaction was flagged
- **Model validation**: confirming model logic aligns with fraud domain knowledge
- **Audit & compliance**: providing interpretable evidence for model decisions

In [ ]:
print('Computing SHAP values...')
explainer   = shap.TreeExplainer(xgb_model)
X_sample    = X_test.sample(min(2000, len(X_test)), random_state=RANDOM_STATE)
shap_values = explainer(X_sample)

# Global feature importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('SHAP Feature Importance — XGBoost Fraud Model\n(red = pushes toward fraud, blue = pushes toward legitimate)',
          fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Mean |SHAP| bar chart (feature ranking)
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=20, show=False)
plt.title('Mean |SHAP| — Feature Importance Ranking', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall plot for the highest-scored fraud case
# Shows exactly WHY the model flagged this specific transaction
fraud_indices = X_sample.index[y_test.loc[X_sample.index].values == 1]
if len(fraud_indices) > 0:
    # Get highest-probability fraud case
    idx_in_sample = [i for i, idx in enumerate(X_sample.index) if idx in fraud_indices]
    top_fraud_idx = max(idx_in_sample, key=lambda i: xgb_probs[X_sample.index[i]])

    plt.figure(figsize=(10, 7))
    shap.waterfall_plot(shap_values[top_fraud_idx], max_display=15, show=False)
    plt.title('SHAP Waterfall — Highest-Confidence Fraud Case\n(How does this transaction score?)',
              fontsize=11, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

    fraud_score = xgb_probs[X_sample.index[top_fraud_idx]]
    print(f'Transaction fraud probability : {fraud_score:.4f} ({fraud_score*100:.2f}%)')

---
## 7. Score Distribution & Model Stability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distributions
fraud_scores = xgb_probs[y_test.values == 1]
legit_scores = xgb_probs[y_test.values == 0]
bins = np.linspace(0, 1, 60)

axes[0].hist(legit_scores, bins=bins, alpha=0.6, color='#457B9D',
             label=f'Legitimate (n={len(legit_scores):,})', density=True)
axes[0].hist(fraud_scores, bins=bins, alpha=0.7, color='#E63946',
             label=f'Fraud (n={len(fraud_scores):,})', density=True)
axes[0].axvline(THRESHOLD, color='#F4A261', lw=2.5, linestyle='--',
                label=f'Threshold = {THRESHOLD}')
axes[0].set_xlabel('XGBoost Fraud Score', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Score Distribution: Fraud vs. Legitimate', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Score decile fraud rate (lift chart)
score_df = pd.DataFrame({'score': xgb_probs, 'is_fraud': y_test.values})
score_df['decile'] = pd.qcut(score_df['score'], q=10, labels=False, duplicates='drop') + 1
decile_stats = score_df.groupby('decile').agg(
    fraud_rate=('is_fraud', 'mean'),
    count=('is_fraud', 'count')
).reset_index()
overall_rate = y_test.mean()
decile_stats['lift'] = decile_stats['fraud_rate'] / overall_rate

bars = axes[1].bar(decile_stats['decile'].astype(str), decile_stats['lift'],
                   color=['#E63946' if l > 1 else '#457B9D' for l in decile_stats['lift']],
                   alpha=0.85)
axes[1].axhline(1.0, color='black', linestyle='--', lw=1.5, label='No lift (=1.0)')
for bar, lift in zip(bars, decile_stats['lift']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{lift:.1f}x', ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Score Decile (1=Lowest Risk, 10=Highest Risk)', fontsize=10)
axes[1].set_ylabel('Lift over Base Rate', fontsize=11)
axes[1].set_title('Score Decile Lift Chart', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Decile lift table:')
display(decile_stats.sort_values('decile', ascending=False).reset_index(drop=True))

---
## 8. Business Impact Summary

Translating model performance into financial and operational impact for stakeholder communication.

In [ ]:
y_pred      = (xgb_probs >= THRESHOLD).astype(int)

fraud_mask  = y_test.values == 1
tp_mask     = (y_pred == 1) & fraud_mask
fn_mask     = (y_pred == 0) & fraud_mask

total_fraud_usd   = amounts_test.values[fraud_mask].sum()
caught_fraud_usd  = amounts_test.values[tp_mask].sum()
escaped_fraud_usd = amounts_test.values[fn_mask].sum()

# Assumptions
recovery_rate         = 0.85  # % of caught fraud recovered/blocked
fp_review_cost_per    = 10    # $10 per false positive alert reviewed
total_fps             = int((y_pred == 1) & (y_test.values == 0)).sum() if False else int(((y_pred==1) & (y_test.values==0)).sum())

fraud_prevented_usd   = caught_fraud_usd  * recovery_rate
fp_review_cost_total  = total_fps * fp_review_cost_per
net_benefit           = fraud_prevented_usd - fp_review_cost_total

print('='*55)
print('  MONTHLY BUSINESS IMPACT SUMMARY (test set)')
print('='*55)
print(f'  Total fraud exposure     : ${total_fraud_usd:>12,.2f}')
print(f'  Fraud caught by model    : ${caught_fraud_usd:>12,.2f}  ({caught_fraud_usd/max(total_fraud_usd,1)*100:.1f}%)')
print(f'  Fraud escaped model      : ${escaped_fraud_usd:>12,.2f}  ({escaped_fraud_usd/max(total_fraud_usd,1)*100:.1f}%)')
print(f'  Estimated loss prevented : ${fraud_prevented_usd:>12,.2f}  (at {recovery_rate*100:.0f}% recovery rate)')
print(f'  False positive alerts    : {total_fps:>13,}')
print(f'  FP review cost           : ${fp_review_cost_total:>12,.2f}  (at ${fp_review_cost_per}/review)')
print(f'  Net financial benefit    : ${net_benefit:>12,.2f}')
print('='*55)
print(f'  ROI of model deployment  : {net_benefit/max(fp_review_cost_total,1):.1f}x return on review cost')

---
## 9. Recommendations & Next Steps

Based on this analysis, the following actions are recommended:

**Model Performance:**
- The XGBoost model achieves strong PR-AUC, substantially outperforming the logistic regression baseline. The model is recommended for production deployment at the 0.50 threshold.
- Consider a **dual-threshold** strategy: auto-decline at score ≥ 0.80, review queue at 0.50–0.79, to reduce analyst workload.

**False Positive Reduction:**
- FP analysis (`sql/04_false_positive_analysis.sql`) shows FPs concentrate in the **online marketplace** and **electronics** merchant categories with scores in the 500–650 range.
- Recommendation: Apply a merchant-category override to reduce threshold to 0.60 for these segments, or add customer tenure as a suppression rule for accounts >2 years old.

**Behavioral Biometrics Integration:**
- `bot_signal` and `new_device_high_ip_risk` are among the top SHAP contributors, validating the value of behavioral and device intelligence integration.
- Next step: Work with the BioCatch and LexisNexis ThreatMetrix teams to pull real-time session features (currently batch-refreshed hourly).

**Model Monitoring:**
- Schedule monthly PSI monitoring (`sql/03_model_kpi_report.sql`, Part 4) to detect score distribution drift.
- Set alert triggers: DR drops >3 percentage points or FPR rises >0.5 percentage points month-over-month.